In [46]:
import pandas as pd
import numpy as np
import glob

In [2]:
df = pd.DataFrame({
    'Dept'  :['Eng','Eng','HR','HR','Mktg','Mktg','Eng'],
    'Name'  :['Alice','Bob','Carol','Dave','Eve','Frank','Grace'],
    'Salary':[90000,85000,60000,65000,70000,72000,95000],
    'Bonus' :[9000,8500,3000,3250,3500,3600,9500]
})

In [3]:
grouped = df.groupby('Dept')
type(grouped)

pandas.api.typing.DataFrameGroupBy

In [4]:
grouped['Salary'].mean()

Dept
Eng     90000.0
HR      62500.0
Mktg    71000.0
Name: Salary, dtype: float64

In [6]:
df.groupby('Dept')['Salary'].mean()

Dept
Eng     90000.0
HR      62500.0
Mktg    71000.0
Name: Salary, dtype: float64

In [7]:
df.groupby('Dept')['Salary'].sum()

Dept
Eng     270000
HR      125000
Mktg    142000
Name: Salary, dtype: int64

In [8]:
df.groupby('Dept')['Salary'].count()

Dept
Eng     3
HR      2
Mktg    2
Name: Salary, dtype: int64

In [9]:
df.groupby('Dept')['Salary'].max()

Dept
Eng     95000
HR      65000
Mktg    72000
Name: Salary, dtype: int64

In [10]:
df.groupby('Dept').size()

Dept
Eng     3
HR      2
Mktg    2
dtype: int64

In [11]:
df.groupby(['Dept', 'Name'])['Salary'].sum()

Dept  Name 
Eng   Alice    90000
      Bob      85000
      Grace    95000
HR    Carol    60000
      Dave     65000
Mktg  Eve      70000
      Frank    72000
Name: Salary, dtype: int64

In [12]:
for dept, group_df in df.groupby('Dept'):
    print(f"\n{dept}:")
    print(group_df)


Eng:
  Dept   Name  Salary  Bonus
0  Eng  Alice   90000   9000
1  Eng    Bob   85000   8500
6  Eng  Grace   95000   9500

HR:
  Dept   Name  Salary  Bonus
2   HR  Carol   60000   3000
3   HR   Dave   65000   3250

Mktg:
   Dept   Name  Salary  Bonus
4  Mktg    Eve   70000   3500
5  Mktg  Frank   72000   3600


In [13]:
eng_group = df.groupby('Dept').get_group('Eng')

In [14]:
df.groupby('Dept').mean(numeric_only=True)

,Salary,Bonus
Dept,,
Eng,90000.0,9000.0
HR,62500.0,3125.0
Mktg,71000.0,3550.0


In [15]:
df.groupby('Dept')['Salary'].agg(['mean', 'min', 'max', 'std', 'count'])

,mean,min,max,std,count
Dept,,,,,
Eng,90000.0,85000,95000,5000.000000,3
HR,62500.0,60000,65000,3535.533906,2
Mktg,71000.0,70000,72000,1414.213562,2


In [16]:
df.groupby('Dept').agg({
    'Salary': ['mean', 'max'],
    'Bonus' : 'sum',
    'Name'  : 'count'
})

Salary         Bonus  Name
         mean    max    sum count
Dept                             
Eng   90000.0  95000  27000     3
HR    62500.0  65000   6250     2
Mktg  71000.0  72000   7100     2

In [17]:
df.groupby('Dept').agg(
    avg_salary  = ('Salary', 'mean'),
    max_salary  = ('Salary', 'max'),
    total_bonus = ('Bonus', 'sum'),
    headcount   = ('Name', 'count')
).reset_index()

,Dept,avg_salary,max_salary,total_bonus,headcount
0,Eng,90000.0,95000,27000,3
1,HR,62500.0,65000,6250,2
2,Mktg,71000.0,72000,7100,2


In [18]:
df.groupby('Dept')['Salary'].agg(lambda x: x.max() - x.min())

Dept
Eng     10000
HR       5000
Mktg     2000
Name: Salary, dtype: int64

In [20]:
df['DeptAvgSalary'] = df.groupby('Dept')['Salary'].transform('mean')
df[['Name','Dept','Salary','DeptAvgSalary']]

,Name,Dept,Salary,DeptAvgSalary
0,Alice,Eng,90000,90000.0
1,Bob,Eng,85000,90000.0
2,Carol,HR,60000,62500.0
3,Dave,HR,65000,62500.0
4,Eve,Mktg,70000,71000.0
5,Frank,Mktg,72000,71000.0
6,Grace,Eng,95000,90000.0


In [21]:
df['SalaryVsDeptAvg'] = df['Salary'] - df['DeptAvgSalary']

In [22]:
big_depts = df.groupby('Dept').filter(lambda g: len(g) >= 2)

In [23]:
def top_earner(group):
    return group.nlargest(1, 'Salary')

In [24]:
top_per_dept = df.groupby('Dept').apply(top_earner)

In [25]:
customers = pd.DataFrame({
    'CustID': [1, 2, 3, 4],
    'Name'  : ['Alice','Bob','Carol','Dave']
})

In [26]:
orders = pd.DataFrame({
    'OrderID': [101,102,103,104],
    'CustID' : [1, 2, 2, 5],
    'Amount' : [500, 750, 300, 200]
})

In [27]:
pd.merge(customers, orders, on='CustID', how='inner')

,CustID,Name,OrderID,Amount
0,1,Alice,101,500
1,2,Bob,102,750
2,2,Bob,103,300


In [28]:
pd.merge(customers, orders, on='CustID', how='left')

,CustID,Name,OrderID,Amount
0,1,Alice,101.0,500.0
1,2,Bob,102.0,750.0
2,2,Bob,103.0,300.0
3,3,Carol,NaN,NaN
4,4,Dave,NaN,NaN


In [29]:
pd.merge(customers, orders, on='CustID', how='right')

,CustID,Name,OrderID,Amount
0,1,Alice,101,500
1,2,Bob,102,750
2,2,Bob,103,300
3,5,NaN,104,200


In [30]:
pd.merge(customers, orders, on='CustID', how='outer')

,CustID,Name,OrderID,Amount
0,1,Alice,101.0,500.0
1,2,Bob,102.0,750.0
2,2,Bob,103.0,300.0
3,3,Carol,NaN,NaN
4,4,Dave,NaN,NaN
5,5,NaN,104.0,200.0


In [31]:
pd.merge(customers, orders, left_on='CustID', right_on='CustID')

,CustID,Name,OrderID,Amount
0,1,Alice,101,500
1,2,Bob,102,750
2,2,Bob,103,300


In [33]:
merged = pd.merge(customers, orders, on='CustID', how='outer', indicator=True)

In [34]:
merged['_merge'].value_counts()

_merge
both          3
left_only     2
right_only    1
Name: count, dtype: int64

In [35]:
df1 = pd.DataFrame({'A':[1,2,3]}, index=['x','y','z'])
df2 = pd.DataFrame({'B':[10,20]}, index=['x','y'])

In [36]:
df1.join(df2)

,A,B
x,1,10.0
y,2,20.0
z,3,NaN


In [37]:
df1.join(df2, how='inner')

,A,B
x,1,10
y,2,20


In [38]:
jan = pd.DataFrame({'Product':['A','B'], 'Sales':[100,200]})
feb = pd.DataFrame({'Product':['A','B'], 'Sales':[150,180]})

In [40]:
combined = pd.concat([jan, feb], ignore_index=True)
combined

,Product,Sales
0,A,100
1,B,200
2,A,150
3,B,180


In [41]:
combined2 = pd.concat([jan, feb], keys=['Jan','Feb'])
combined2

Product  Sales
Jan 0       A    100
    1       B    200
Feb 0       A    150
    1       B    180

In [42]:
df_a = pd.DataFrame({'X':[1,2,3]})
df_b = pd.DataFrame({'Y':[4,5,6]})
pd.concat([df_a, df_b], axis=1)

,X,Y
0,1,4
1,2,5
2,3,6


In [43]:
df_x = pd.DataFrame({'A':[1,2], 'B':[3,4]})
df_y = pd.DataFrame({'B':[5,6], 'C':[7,8]})

In [44]:
pd.concat([df_x, df_y])


,A,B,C
0,1.0,3,NaN
1,2.0,4,NaN
0,NaN,5,7.0
1,NaN,6,8.0


In [45]:
pd.concat([df_x, df_y], join='inner')

,B
0,3
1,4
0,5
1,6


In [48]:
all_files = glob.glob('sales_2024_*.csv')
# all_months = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

In [49]:
sales = pd.DataFrame({
    'Region'  :['North','North','South','South','North','South'],
    'Product' :['A','B','A','B','A','A'],
    'Quarter' :['Q1','Q1','Q1','Q1','Q2','Q2'],
    'Revenue' :[1000,1500,800,1200,1100,900]
})

In [50]:
pivot = pd.pivot_table(sales,
                       values='Revenue',
                       index='Region',
                       columns='Product',
                       aggfunc='sum')
pivot

Product,A,B
Region,,
North,2100,1500
South,1700,1200


In [51]:
pd.pivot_table(sales, values='Revenue',
                index=['Region','Quarter'], columns='Product', aggfunc='sum')

Product              A       B
Region Quarter                
North  Q1       1000.0  1500.0
       Q2       1100.0     NaN
South  Q1        800.0  1200.0
       Q2        900.0     NaN

In [52]:
pd.pivot_table(sales, values='Revenue', index='Region',
                columns='Product', aggfunc=['sum','mean','count'])

sum          mean         count   
Product     A     B       A       B     A  B
Region                                      
North    2100  1500  1050.0  1500.0     2  1
South    1700  1200   850.0  1200.0     2  1

In [53]:
pd.pivot_table(sales, values='Revenue', index='Region',
                columns='Product', aggfunc='sum', fill_value=0)

Product,A,B
Region,,
North,2100,1500
South,1700,1200


In [54]:
pd.pivot_table(sales, values='Revenue', index='Region',
                columns='Product', aggfunc='sum', margins=True, margins_name='Total')

Product,A,B,Total
Region,,,
North,2100,1500,3600
South,1700,1200,2900
Total,3800,2700,6500


In [56]:
wide_df = pd.DataFrame({
    'Name': ['Alice', 'Bob'],
    'Q1'  : [100, 150],
    'Q2'  : [120, 130],
    'Q3'  : [140, 160],
    'Q4'  : [160, 170]
})

In [58]:
long_df = pd.melt(wide_df,
                  id_vars='Name',
                  value_vars=['Q1','Q2','Q3','Q4'],
                  var_name='Quarter',     
                  value_name='Revenue') 
long_df

,Name,Quarter,Revenue
0,Alice,Q1,100
1,Bob,Q1,150
2,Alice,Q2,120
3,Bob,Q2,130
4,Alice,Q3,140
5,Bob,Q3,160
6,Alice,Q4,160
7,Bob,Q4,170


In [59]:
pd.melt(wide_df, id_vars='Name')

,Name,variable,value
0,Alice,Q1,100
1,Bob,Q1,150
2,Alice,Q2,120
3,Bob,Q2,130
4,Alice,Q3,140
5,Bob,Q3,160
6,Alice,Q4,160
7,Bob,Q4,170


In [60]:
back_to_wide = long_df.pivot(index='Name', columns='Quarter', values='Revenue')
back_to_wide

Quarter,Q1,Q2,Q3,Q4
Name,,,,
Alice,100,120,140,160
Bob,150,130,160,170


In [61]:
back_to_wide.reset_index()

Quarter,Name,Q1,Q2,Q3,Q4
0,Alice,100,120,140,160
1,Bob,150,130,160,170


In [62]:
df = pd.DataFrame({'Q1':[100,150], 'Q2':[120,130]},
                  index=['Alice','Bob'])

stacked = df.stack()
stacked

Alice  Q1    100
       Q2    120
Bob    Q1    150
       Q2    130
dtype: int64

In [63]:
unstacked = stacked.unstack()
unstacked

,Q1,Q2
Alice,100,120
Bob,150,130


In [64]:
sales_grouped = sales.groupby(['Region','Product'])['Revenue'].sum()
sales_grouped.unstack()

Product,A,B
Region,,
North,2100,1500
South,1700,1200
